In [ ]:
import requests
import pandas as pd
import time
import os

BASE = "https://api.openf1.org/v1"
os.makedirs("raw", exist_ok=True)

#OpenF1 free: 30 запросов в минуту -пауза 2.5 сек 
def get_json(endpoint, params, retries=3):
    url = f"{BASE}/{endpoint}"
    for attempt in range(retries):
        try:
            response = requests.get(url, params=params, timeout=30)
            response.raise_for_status()
            time.sleep(2.5)
            return response.json()
        #ВЕРНУТТСЯ ЧЕКНУТЬ -по ошибкам из семинара брал, но думаю нам тоже подойдет
        except (requests.exceptions.SSLError, requests.exceptions.ConnectionError, requests.exceptions.Timeout): #ОБЬЯСНЮ ПРИ ВСТРЕЧЕ но тут все из
            #если сетевой сбой -ждем дольше и пробуем снова
            print(f"  сетевая ошибка, повтор {attempt+1}/{retries}...")
            time.sleep(5)
    #если все попытки провалились, пробрасываем ошибку
    raise Exception(f"не удалось получить {endpoint} после {retries} попыток")

In [ ]:
SEASONS = [2023, 2024, 2025]

all_sessions = []
for year in SEASONS:
    data = get_json("sessions", {"year": year})
    all_sessions.append(pd.DataFrame(data))

sessions = pd.concat(all_sessions, ignore_index=True)
sessions.to_csv("raw/sessions.csv", index=False)

#только основные гонки и квалы (без практик,тестов, спринтов)
races = sessions[sessions["session_name"] == "Race"].copy()
qualis = sessions[sessions["session_name"] == "Qualifying"].copy()

print("гонок:", len(races), "квал:", len(qualis))

гонок: 71 квал: 71


In [ ]:
def get_best_quali_time(duration):
    #в квале duration это список [Q1, Q2, Q3], берем лучшее непустое время
    if isinstance(duration, list):
        times = [t for t in duration if t is not None]
        if len(times) > 0:
            return min(times)
        return None
    return duration


def collect_session_results(session_key, session_type):
    data = get_json("session_result", {"session_key": session_key})
    df = pd.DataFrame(data)
    if len(df) == 0:
        return df

    keep = ["session_key", "meeting_key", "driver_number", "position","number_of_laps", "dnf", "dns", "dsq", "gap_to_leader"]

    if "points" in df.columns:
        keep.append("points")
    result = df[keep].copy()








    #время:в квале лучшее из Q1,Q2,Q3, в гонке duration как есть
    if session_type == "Qualifying":
        result["best_time"] = df["duration"].apply(get_best_quali_time)
    else:
        result["best_time"] = df["duration"]

    return result

In [38]:
def collect_all_results(sessions_df, session_type):
    all_results = []
    failed = []
    keys = sessions_df["session_key"].tolist()

    for i, key in enumerate(keys):
        try:
            one = collect_session_results(key, session_type)
            if len(one) > 0:
                all_results.append(one)
                print(f"ок [{i+1}/{len(keys)}] {session_type} {key}, строк {len(one)}")
            else:
                failed.append(key)
                print(f"пусто: {session_type} {key}")
        except Exception as e:
            failed.append(key)
            print(f"пропуск: {session_type} {key} -> {type(e).__name__}")

    result = pd.concat(all_results, ignore_index=True)
    return result, failed

In [39]:
quali_results, quali_failed = collect_all_results(qualis, "Qualifying")
quali_results.to_csv("raw/quali_results.csv", index=False)
print("КВАЛЫ:", quali_results.shape, "не собралось:", len(quali_failed))

ок [1/71] Qualifying 7768, строк 20
ок [2/71] Qualifying 7775, строк 20
ок [3/71] Qualifying 7783, строк 20
ок [4/71] Qualifying 9064, строк 20
ок [5/71] Qualifying 9074, строк 20
пропуск: Qualifying 9082 -> HTTPError
ок [7/71] Qualifying 9090, строк 20
ок [8/71] Qualifying 9098, строк 20
ок [9/71] Qualifying 9106, строк 20
ок [10/71] Qualifying 9112, строк 20
ок [11/71] Qualifying 9122, строк 20
ок [12/71] Qualifying 9129, строк 20
пропуск: Qualifying 9135 -> HTTPError
ок [14/71] Qualifying 9145, строк 20
ок [15/71] Qualifying 9153, строк 20
ок [16/71] Qualifying 9161, строк 20
ок [17/71] Qualifying 9169, строк 20
ок [18/71] Qualifying 9215, строк 20
ок [19/71] Qualifying 9207, строк 20
ок [20/71] Qualifying 9177, строк 20
ок [21/71] Qualifying 9304, строк 20
ок [22/71] Qualifying 9314, строк 20
ок [23/71] Qualifying 9193, строк 20
ок [24/71] Qualifying 9468, строк 20
ок [25/71] Qualifying 9476, строк 20
ок [26/71] Qualifying 9484, строк 19
ок [27/71] Qualifying 9492, строк 20
ок [28/

сбор пилотов и команд

In [ ]:
def collect_drivers(session_key):
    data = get_json("drivers", {"session_key": session_key})
    df = pd.DataFrame(data)
    if len(df) == 0:
        return df
    keep = ["session_key", "driver_number", "full_name", "name_acronym", "team_name"]
    return df[keep].copy()


# собираем илотов по всем гонкам (команды могут меняться, поэтому по каждрй сессии)
all_drivers = []
race_keys = races["session_key"].tolist()

for i, key in enumerate(race_keys):
    try:
        one = collect_drivers(key)
        if len(one) > 0:
            all_drivers.append(one)
            print(f"ок [{i+1}/{len(race_keys)}] drivers {key}, строк {len(one)}")
    except Exception as e:
        print(f"пропуск: drivers {key} -> {type(e).__name__}")

drivers = pd.concat(all_drivers, ignore_index=True)
drivers.to_csv("raw/drivers.csv", index=False)
print("ВСЕГО пилотов:", drivers.shape)

ок [1/71] drivers 7953, строк 20
ок [2/71] drivers 7779, строк 20
ок [3/71] drivers 7787, строк 20
ок [4/71] drivers 9070, строк 20
ок [5/71] drivers 9078, строк 20
пропуск: drivers 9086 -> HTTPError
ок [7/71] drivers 9094, строк 20
ок [8/71] drivers 9102, строк 20
ок [9/71] drivers 9110, строк 20
ок [10/71] drivers 9118, строк 20
ок [11/71] drivers 9126, строк 20
ок [12/71] drivers 9133, строк 20
ок [13/71] drivers 9141, строк 20
ок [14/71] drivers 9149, строк 20
ок [15/71] drivers 9157, строк 20
ок [16/71] drivers 9165, строк 19
ок [17/71] drivers 9173, строк 20
ок [18/71] drivers 9221, строк 20
ок [19/71] drivers 9213, строк 20
ок [20/71] drivers 9181, строк 20
ок [21/71] drivers 9205, строк 20
ок [22/71] drivers 9189, строк 20
ок [23/71] drivers 9197, строк 20
ок [24/71] drivers 9472, строк 20
ок [25/71] drivers 9480, строк 20
  сетевая ошибка, повтор 1/3...
  сетевая ошибка, повтор 2/3...
ок [26/71] drivers 9488, строк 19
ок [27/71] drivers 9496, строк 20
ок [28/71] drivers 9673, 

сбор стартовой решетки

In [ ]:
def collect_weather(session_key):
    data = get_json("weather", {"session_key": session_key})
    df = pd.DataFrame(data)
    if len(df) == 0:
        return None



    # погода меняется в течение сессии, берем средние значения за гонку
    weather_row = {
        "session_key": session_key,
        "air_temp_mean": df["air_temperature"].mean(),
        "track_temp_mean": df["track_temperature"].mean(),
        "humidity_mean": df["humidity"].mean(),
        "wind_speed_mean": df["wind_speed"].mean(),
        "rainfall_max": df["rainfall"].max()}# был ли дождь хоть раз
    return pd.DataFrame([weather_row])


all_weather = []
race_keys = races["session_key"].tolist()

for i, key in enumerate(race_keys):
    try:
        one = collect_weather(key)
        if one is not None:
            all_weather.append(one)
            print(f"ок [{i+1}/{len(race_keys)}] weather {key}")
        else:
            print(f"пусто: weather {key}")
    except Exception as e:
        print(f"пропуск: weather {key} -> {type(e).__name__}")

weather = pd.concat(all_weather, ignore_index=True)
weather.to_csv("raw/weather.csv", index=False)
print("ВСЕГО погоды:", weather.shape)

ок [1/71] weather 7953
ок [2/71] weather 7779
ок [3/71] weather 7787
ок [4/71] weather 9070
ок [5/71] weather 9078
пропуск: weather 9086 -> HTTPError
ок [7/71] weather 9094
ок [8/71] weather 9102
ок [9/71] weather 9110
ок [10/71] weather 9118
ок [11/71] weather 9126
ок [12/71] weather 9133
ок [13/71] weather 9141
ок [14/71] weather 9149
ок [15/71] weather 9157
ок [16/71] weather 9165
ок [17/71] weather 9173
ок [18/71] weather 9221
ок [19/71] weather 9213
ок [20/71] weather 9181
ок [21/71] weather 9205
ок [22/71] weather 9189
ок [23/71] weather 9197
ок [24/71] weather 9472
ок [25/71] weather 9480
ок [26/71] weather 9488
ок [27/71] weather 9496
ок [28/71] weather 9673
ок [29/71] weather 9507
ок [30/71] weather 9515
ок [31/71] weather 9523
ок [32/71] weather 9531
ок [33/71] weather 9539
ок [34/71] weather 9550
ок [35/71] weather 9558
ок [36/71] weather 9566
ок [37/71] weather 9574
ок [38/71] weather 9582
ок [39/71] weather 9590
ок [40/71] weather 9598
ок [41/71] weather 9606
ок [42/71] we

In [ ]:
def collect_meetings(year):
    data = get_json("meetings", {"year": year})
    df = pd.DataFrame(data)
    keep = ["meeting_key", "year", "country_name", "location","circuit_short_name", "circuit_type", "meeting_name"]
    return df[keep].copy()


all_meetings = []
for year in SEASONS:
    one = collect_meetings(year)
    all_meetings.append(one)
    print(f"ок {year}: трасс {len(one)}")

meetings = pd.concat(all_meetings, ignore_index=True)
meetings.to_csv("raw/meetings.csv", index=False)
print("ВСЕГО трасс:", meetings.shape)
meetings.head(10)

ок 2023: трасс 24
ок 2024: трасс 25
ок 2025: трасс 25
ВСЕГО трасс: (74, 7)


,meeting_key,year,country_name,location,circuit_short_name,circuit_type,meeting_name
0,1140,2023,Bahrain,Sakhir,Sakhir,Permanent,Pre-Season Testing
1,1141,2023,Bahrain,Sakhir,Sakhir,Permanent,Bahrain Grand Prix
2,1142,2023,Saudi Arabia,Jeddah,Jeddah,Temporary - Street,Saudi Arabian Grand Prix
3,1143,2023,Australia,Melbourne,Melbourne,Temporary - Street,Australian Grand Prix
4,1207,2023,Azerbaijan,Baku,Baku,Temporary - Street,Azerbaijan Grand Prix
5,1208,2023,United States,Miami,Miami,Temporary - Street,Miami Grand Prix
6,1209,2023,Italy,Imola,Imola,Permanent,Emilia Romagna Grand Prix
7,1210,2023,Monaco,Monaco,Monte Carlo,Temporary - Street,Monaco Grand Prix
8,1211,2023,Spain,Barcelona,Catalunya,Permanent,Spanish Grand Prix
9,1212,2023,Canada,Montréal,Montreal,Permanent,Canadian Grand Prix


In [ ]:
def collect_laps_agg(session_key):
    data = get_json("laps", {"session_key": session_key})
    df = pd.DataFrame(data)
    if len(df) == 0:
        return None

    #ОБЬЯСНЮ ПРИ ВСТРЕЧЕ агрегируем по пилоту: из всех кругов гонки делаем одну строку на пилота
    grouped = df.groupby("driver_number").agg(
        best_lap=("lap_duration", "min"),
        avg_lap=("lap_duration", "mean"),
        avg_st_speed=("st_speed", "mean"),
        max_st_speed=("st_speed", "max"),
        avg_i1_speed=("i1_speed", "mean"),
        avg_i2_speed=("i2_speed", "mean"),
        laps_count=("lap_number", "count"),
    ).reset_index()

    grouped["session_key"] = session_key
    return grouped


all_laps = []
race_keys = races["session_key"].tolist()

for i, key in enumerate(race_keys):
    try:
        one = collect_laps_agg(key)
        if one is not None:
            all_laps.append(one)
            print(f"ок [{i+1}/{len(race_keys)}] laps {key}, пилотов {len(one)}")
        else:
            print(f"пусто: laps {key}")
    except Exception as e:
        print(f"пропуск: laps {key} -> {type(e).__name__}")

laps_agg = pd.concat(all_laps, ignore_index=True)
laps_agg.to_csv("raw/laps_agg.csv", index=False)
print("ВСЕГО телеметрии:", laps_agg.shape)

ок [1/71] laps 7953, пилотов 20
ок [2/71] laps 7779, пилотов 20
ок [3/71] laps 7787, пилотов 20
ок [4/71] laps 9070, пилотов 20
ок [5/71] laps 9078, пилотов 20
пропуск: laps 9086 -> HTTPError
ок [7/71] laps 9094, пилотов 20
ок [8/71] laps 9102, пилотов 20
ок [9/71] laps 9110, пилотов 20
ок [10/71] laps 9118, пилотов 20
ок [11/71] laps 9126, пилотов 20
ок [12/71] laps 9133, пилотов 20
ок [13/71] laps 9141, пилотов 20
ок [14/71] laps 9149, пилотов 20
ок [15/71] laps 9157, пилотов 20
ок [16/71] laps 9165, пилотов 19
ок [17/71] laps 9173, пилотов 20
ок [18/71] laps 9221, пилотов 19
ок [19/71] laps 9213, пилотов 20
ок [20/71] laps 9181, пилотов 20
ок [21/71] laps 9205, пилотов 20
ок [22/71] laps 9189, пилотов 20
ок [23/71] laps 9197, пилотов 20
ок [24/71] laps 9472, пилотов 20
ок [25/71] laps 9480, пилотов 20
ок [26/71] laps 9488, пилотов 19
ок [27/71] laps 9496, пилотов 20
ок [28/71] laps 9673, пилотов 20
ок [29/71] laps 9507, пилотов 20
ок [30/71] laps 9515, пилотов 20
ок [31/71] laps 952

/var/folders/p9/qwbt06j908qdt779l1284brh0000gn/T/ipykernel_12368/2354936556.py:36: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  laps_agg = pd.concat(all_laps, ignore_index=True)


In [ ]:
import pandas as pd
import numpy as np

#грузим все собранное
race_results = pd.read_csv("raw/race_results.csv")
quali_results = pd.read_csv("raw/quali_results.csv")
drivers = pd.read_csv("raw/drivers.csv")
weather = pd.read_csv("raw/weather.csv")
meetings = pd.read_csv("raw/meetings.csv")
laps_agg = pd.read_csv("raw/laps_agg.csv")
sessions = pd.read_csv("raw/sessions.csv")

print("результаты гонок:", race_results.shape)
print("результаты квал:", quali_results.shape)
print("пилоты:", drivers.shape)
print("погода:", weather.shape)
print("трассы:", meetings.shape)
print("телеметрия:", laps_agg.shape)

результаты гонок: (1397, 11)
результаты квал: (1377, 10)
пилоты: (1397, 5)
погода: (70, 6)
трассы: (74, 7)
телеметрия: (1394, 9)


In [ ]:
# основа: результаты гонок
race = race_results.copy()

# 1)добавляем пилотов и команды (по session_key + driver_number)
race = race.merge(drivers, on=["session_key", "driver_number"], how="left")

# 2)добавляем телеметрию (по session_key + driver_number)
race = race.merge(laps_agg, on=["session_key", "driver_number"], how="left")

#3) добавляем погоду (по session_key)
race = race.merge(weather, on="session_key", how="left")

#4)добавляем инфу о трассе (по meeting_key)
track_cols = ["meeting_key", "country_name", "circuit_short_name", "circuit_type", "year"]
race = race.merge(meetings[track_cols], on="meeting_key", how="left")
# тут тоже сложноватый мердж, обьясню при встрече, но я хз как еще там
print("итоговая таблица гонки:", race.shape)
print("колонки:", list(race.columns))
race.head(5)

итоговая таблица гонки: (1397, 30)
колонки: ['session_key', 'meeting_key', 'driver_number', 'position', 'number_of_laps', 'dnf', 'dns', 'dsq', 'gap_to_leader', 'points', 'best_time', 'full_name', 'name_acronym', 'team_name', 'best_lap', 'avg_lap', 'avg_st_speed', 'max_st_speed', 'avg_i1_speed', 'avg_i2_speed', 'laps_count', 'air_temp_mean', 'track_temp_mean', 'humidity_mean', 'wind_speed_mean', 'rainfall_max', 'country_name', 'circuit_short_name', 'circuit_type', 'year']


,session_key,meeting_key,driver_number,position,number_of_laps,dnf,dns,dsq,gap_to_leader,points,...,laps_count,air_temp_mean,track_temp_mean,humidity_mean,wind_speed_mean,rainfall_max,country_name,circuit_short_name,circuit_type,year
0,7953,1141,1,1.0,57.0,False,False,False,0,25.0,...,57.0,27.431677,31.011801,21.496894,0.68323,0,Bahrain,Sakhir,Permanent,2023
1,7953,1141,11,2.0,57.0,False,False,False,11.987,18.0,...,57.0,27.431677,31.011801,21.496894,0.68323,0,Bahrain,Sakhir,Permanent,2023
2,7953,1141,14,3.0,57.0,False,False,False,38.637,15.0,...,57.0,27.431677,31.011801,21.496894,0.68323,0,Bahrain,Sakhir,Permanent,2023
3,7953,1141,55,4.0,57.0,False,False,False,48.052,12.0,...,57.0,27.431677,31.011801,21.496894,0.68323,0,Bahrain,Sakhir,Permanent,2023
4,7953,1141,44,5.0,57.0,False,False,False,50.977,10.0,...,57.0,27.431677,31.011801,21.496894,0.68323,0,Bahrain,Sakhir,Permanent,2023


In [49]:
# основа: результаты квал
quali = quali_results.copy()

# добавляем пилотов и команды
quali = quali.merge(drivers, on=["session_key", "driver_number"], how="left")

# добавляем погоду
quali = quali.merge(weather, on="session_key", how="left")

# добавляем инфу о трассе
quali = quali.merge(meetings[track_cols], on="meeting_key", how="left")

print("итоговая таблица квалы:", quali.shape)
print("колонки:", list(quali.columns))
quali.head(5)

итоговая таблица квалы: (1377, 22)
колонки: ['session_key', 'meeting_key', 'driver_number', 'position', 'number_of_laps', 'dnf', 'dns', 'dsq', 'gap_to_leader', 'best_time', 'full_name', 'name_acronym', 'team_name', 'air_temp_mean', 'track_temp_mean', 'humidity_mean', 'wind_speed_mean', 'rainfall_max', 'country_name', 'circuit_short_name', 'circuit_type', 'year']


,session_key,meeting_key,driver_number,position,number_of_laps,dnf,dns,dsq,gap_to_leader,best_time,...,team_name,air_temp_mean,track_temp_mean,humidity_mean,wind_speed_mean,rainfall_max,country_name,circuit_short_name,circuit_type,year
0,7768,1141,1,1.0,15,False,False,False,"[0.302, 0.221, 0.0]",89.708,...,NaN,NaN,NaN,NaN,NaN,NaN,Bahrain,Sakhir,Permanent,2023
1,7768,1141,11,2.0,15,False,False,False,"[0.486, 0.464, 0.138]",89.846,...,NaN,NaN,NaN,NaN,NaN,NaN,Bahrain,Sakhir,Permanent,2023
2,7768,1141,16,3.0,17,False,False,False,"[0.101, 0.0, 0.292]",90.000,...,NaN,NaN,NaN,NaN,NaN,NaN,Bahrain,Sakhir,Permanent,2023
3,7768,1141,55,4.0,18,False,False,False,"[0.0, 0.233, 0.446]",90.154,...,NaN,NaN,NaN,NaN,NaN,NaN,Bahrain,Sakhir,Permanent,2023
4,7768,1141,14,5.0,15,False,False,False,"[0.165, 0.363, 0.628]",90.336,...,NaN,NaN,NaN,NaN,NaN,NaN,Bahrain,Sakhir,Permanent,2023


In [ ]:
# команды берем из drivers, но привязываем по уикенду (meeting_key), А НЕ СЕСИИИ -НАПОМНИТЬ 



# для этого добавим meeting_key в drivers через race_results
driver_team = race_results[["meeting_key", "driver_number"]].merge(
    drivers[["session_key", "driver_number", "team_name", "full_name", "name_acronym"]],
    on="driver_number", how="left")


# оставляем уникальные пары уикенд + пилот + команда
driver_team = driver_team[["meeting_key", "driver_number", "team_name", "full_name", "name_acronym"]].drop_duplicates(subset=["meeting_key", "driver_number"])


# погоду тоже привяжем к уикенду: добавим meeting_key в weather через race_results
weather_meeting = race_results[["session_key", "meeting_key"]].drop_duplicates().merge(weather, on="session_key", how="left")

weather_cols = ["meeting_key", "air_temp_mean", "track_temp_mean", "humidity_mean", "wind_speed_mean", "rainfall_max"]
weather_meeting = weather_meeting[weather_cols].drop_duplicates(subset=["meeting_key"])

#теперь пересобираем квалу, клеим по meeting_key
quali = quali_results.copy()
quali = quali.merge(driver_team, on=["meeting_key", "driver_number"], how="left")
quali = quali.merge(weather_meeting, on="meeting_key", how="left")
quali = quali.merge(meetings[track_cols], on="meeting_key", how="left")

print("квала исправленная:", quali.shape)
print("команды заполнены:", quali["team_name"].notna().sum(), "из", len(quali))
quali.head(5)

квала исправленная: (1377, 22)
команды заполнены: 1375 из 1377


,session_key,meeting_key,driver_number,position,number_of_laps,dnf,dns,dsq,gap_to_leader,best_time,...,name_acronym,air_temp_mean,track_temp_mean,humidity_mean,wind_speed_mean,rainfall_max,country_name,circuit_short_name,circuit_type,year
0,7768,1141,1,1.0,15,False,False,False,"[0.302, 0.221, 0.0]",89.708,...,VER,27.431677,31.011801,21.496894,0.68323,0,Bahrain,Sakhir,Permanent,2023
1,7768,1141,11,2.0,15,False,False,False,"[0.486, 0.464, 0.138]",89.846,...,PER,27.431677,31.011801,21.496894,0.68323,0,Bahrain,Sakhir,Permanent,2023
2,7768,1141,16,3.0,17,False,False,False,"[0.101, 0.0, 0.292]",90.000,...,LEC,27.431677,31.011801,21.496894,0.68323,0,Bahrain,Sakhir,Permanent,2023
3,7768,1141,55,4.0,18,False,False,False,"[0.0, 0.233, 0.446]",90.154,...,SAI,27.431677,31.011801,21.496894,0.68323,0,Bahrain,Sakhir,Permanent,2023
4,7768,1141,14,5.0,15,False,False,False,"[0.165, 0.363, 0.628]",90.336,...,ALO,27.431677,31.011801,21.496894,0.68323,0,Bahrain,Sakhir,Permanent,2023
